In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import math
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_absolute_percentage_error

from loaders._load_vn30_reg_deep import preprocess, VN30, TARGETS
from models.regression.tft import TemporalFusionTransformer

In [6]:
train_loader, valid_loader, test_loader, scaler = preprocess('ACB', 'tft', verbose=True)

Train shape: torch.Size([1094, 30, 4]), torch.Size([1094, 4])
Valid shape: torch.Size([121, 30, 4]), torch.Size([121, 4])


In [7]:
model = TemporalFusionTransformer()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.8)
criterion = nn.SmoothL1Loss()

In [8]:
best_val_loss = float('inf')
n_epochs = 50

for epoch in range(1, n_epochs + 1):
    # --- train ---
    model.train()
    train_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch, y_batch
        optimizer.zero_grad()
        preds = model(X_batch)
        loss  = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * X_batch.size(0)
    train_loss /= len(train_loader.dataset)

    # --- validation ---
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in valid_loader:
            X_batch, y_batch = X_batch, y_batch
            preds = model(X_batch)
            val_loss += criterion(preds, y_batch).item() * X_batch.size(0)
    val_loss /= len(valid_loader.dataset)

    scheduler.step()

    # --- checkpoint ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), f'checkpoints/tft_ACB.pth')

    if epoch % 10 == 0 or epoch == n_epochs:
        print(f"Epoch {epoch:3d}/{n_epochs}: "
              f"Train Loss = {train_loss:.6f}, "
              f"Valid Loss = {val_loss:.6f}, "
              f"Best Val Loss = {best_val_loss:.6f}, "
			  f"LR = {optimizer.param_groups[0]['lr']:.6f}")

Epoch  10/50: Train Loss = 0.001261, Valid Loss = 0.000797, Best Val Loss = 0.000797, LR = 0.000800
Epoch  20/50: Train Loss = 0.001095, Valid Loss = 0.000780, Best Val Loss = 0.000674, LR = 0.000640
Epoch  30/50: Train Loss = 0.001057, Valid Loss = 0.000679, Best Val Loss = 0.000665, LR = 0.000512
Epoch  40/50: Train Loss = 0.001050, Valid Loss = 0.000677, Best Val Loss = 0.000660, LR = 0.000410
Epoch  50/50: Train Loss = 0.001060, Valid Loss = 0.000754, Best Val Loss = 0.000660, LR = 0.000328


In [ ]:
eval_dict = {}

for symbol in VN30:
    eval_dict[symbol] = []

def eval(symbol):
    _, _, test_loader, scaler = preprocess(symbol, 'tft', verbose=True)
    model.load_state_dict(torch.load(f'checkpoints/tft_{symbol}.pth', map_location='cpu'))
    model.eval()

    # Thu thập dự đoán và nhãn
    all_preds   = []
    all_targets = []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch
            preds = model(X_batch).cpu().numpy()
            all_preds.append(preds)
            all_targets.append(y_batch.numpy())

    all_preds   = np.vstack(all_preds)   # (n_samples, 5)
    all_targets = np.vstack(all_targets)

    # Inverse scaling
    all_preds_inv   = scaler.inverse_transform(all_preds)
    all_targets_inv = scaler.inverse_transform(all_targets)

    # Tính metrics
    r2   = r2_score(all_targets_inv, all_preds_inv, multioutput='uniform_average')
    mape = mean_absolute_percentage_error(all_targets_inv, all_preds_inv) * 100

    eval_dict[symbol].append((r2, mape))
    print(f"Test R²: {r2:.4f}")
    print(f"Test MAPE: {mape:.4f}%")